# 13 — submit_21 + 8/25 라벨 규칙 블렌드 (submit_36, 리더보드 0.6696)

**아이디어**: 9/1 테스트는 08-26 이후 조회수 급등 국면이다. 이 국면에서는 공식 **08-25 라벨**이 학습 기간(급등 전)보다 훨씬 강한 신호가 된다.
- 08-25 라벨의 정답 구간(08-25 ~ 09-12)은 9/1 정답 구간(09-01 ~ 09-26)의 앞 12일과 겹친다.
- 급등 국면 안에서 만든 의사 정답(기준일 08-28·08-29)에서 08-25 라벨 규칙이 가장 강했다(PR-AUC 0.49 ~ 0.56).
- 그런데 급등 전 기간으로 학습한 모델(submit_21)은 이 라벨을 여러 피처 중 하나로만 써서 비중이 낮다
  (21 상위 139개 중 08-25 라벨=1은 117개).

→ 모델 점수(submit_21) 순위에 **08-25 라벨 규칙** 순위를 직접 섞어 라벨 비중을 올린다.

| 제출 | 구성 | 리더보드 |
|---|---|---|
| submit_21 | 0.35·s4 + 0.35·s3 + 0.30·submit_15 순위평균 (11_ensemble.ipynb) | 0.6420 |
| **submit_36** | **0.7·rank(21) + 0.3·rank(08-25 라벨 규칙)** | **0.6696** |
| submit_38 | 0.5·rank(21) + 0.5·rank(08-25 라벨 규칙) | — |
| submit_39 | 0.3·rank(21) + 0.7·rank(08-25 라벨 규칙) | — |
| submit_40 | 08-25 라벨=1 그룹 우선, 그룹 안은 21 순서 | — |

> ⚠ **운영진 확인 필요** — 공식 `train_labels.csv`의 08-25 라벨(09-01 이후 영상 성과로 계산됨)을 사용한다.

입력: `submissions/submit_21_ens_s4_s3_s15.csv` (11_ensemble.ipynb 출력), 원본 데이터 `video_5d_views.csv`, `train_labels.csv`

In [1]:
# ===== 라이브러리 · 경로 =====
import os
import duckdb
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from IPython.display import display

RAW = os.environ.get('EPOCH_RAW', '../data/raw/epoch_data')
SUB = os.environ.get('EPOCH_SUB', '../submissions')
D, PREV = '2026-09-01', '2026-08-25'       # 본선 기준일 / 전주 공식 라벨 기준일

## 1. 입력 준비

- `lp` : 9/1 과거 중앙값 = 기준일 이전 롱폼 log(1 + view_5d)의 중앙값 (정답 공식의 past_score와 같은 의미, 원값)
- `prev` : 공식 08-25 라벨 (없는 채널 28개는 기저율 0.2)
- `s21` : submit_21 예측값

In [2]:
con = duckdb.connect()
lp = con.sql(f"""
    SELECT channel_id, MEDIAN(LN(1 + view_5d)) AS lp
    FROM read_csv('{RAW}/video_5d_views.csv', header=true, auto_detect=true)
    WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{D}'
    GROUP BY 1""").df()

# 채점 대상 694행 (row_id = '채널ID_날짜')
sample = pd.read_csv(f'{RAW}/sample_submission.csv', encoding='utf-8-sig')[['row_id']]
sample['channel_id'] = sample.row_id.str.rsplit('_', n=1).str[0]

# 공식 08-25 라벨
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
prev = tl[tl.row_id.str.endswith(PREV)][['channel_id', 'target']].rename(columns={'target': 'prev'})

m = sample.merge(lp, on='channel_id', how='left').merge(prev, on='channel_id', how='left')
m['s21'] = pd.read_csv(f'{SUB}/submit_21_ens_s4_s3_s15.csv').set_index('row_id').prediction.loc[m.row_id].values
print('채널 수:', len(m), '| 08-25 라벨 1 / 0 / 없음:', int((m.prev == 1).sum()), int((m.prev == 0).sum()), int(m.prev.isna().sum()))

채널 수: 694 | 08-25 라벨 1 / 0 / 없음: 134 532 28


## 2. 08-25 라벨 규칙

라벨=1 채널을 위에, 라벨=0 채널을 아래에 둔다(라벨 없는 채널은 기저율 0.2로 중간).
같은 그룹 안에서는 **과거 중앙값이 낮은 채널을 위로** — 정답이 `log(미래) − log(과거)`라 과거가 낮을수록 Rising이 되기 쉽다.

In [3]:
N = len(m)
def rank01(x):
    """값을 0~1 순위로 (결측은 중앙값으로 채움)"""
    x = pd.Series(np.asarray(x, float))
    return rankdata(x.fillna(x.median())) / N

prev_f = m.prev.fillna(0.2).values
r_rule = rank01(prev_f + 1e-3 * rank01(-m.lp))     # 라벨이 우선, 동률 안에서는 과거 중앙값 낮은 순
r21 = rank01(m.s21)

# 21 상위 139개(양성 수 ≈ 694 × 0.2) 중 08-25 라벨=1 채널 수
top21 = np.argsort(-r21)[:139]
print('submit_21 상위 139 중 08-25 라벨=1:', int((m.prev.values[top21] == 1).sum()))

submit_21 상위 139 중 08-25 라벨=1: 117


## 3. 블렌드 · 제출 파일

순위 평균 후, 동점은 21 순위로 깬다. 제출 가이드 §3 검사(694행, row_id 일치, 0~1, 결측·동점 없음)를 통과해야 저장한다.

In [4]:
def save(score, fname, desc):
    s = pd.Series(np.asarray(score, float) + 1e-6 * r21, index=m.row_id.values)    # 동점 깨기: 21 순위
    s = (s - s.min()) / (s.max() - s.min())
    sub = pd.DataFrame({'row_id': s.index, 'prediction': s.values})
    assert list(sub.columns) == ['row_id', 'prediction'] and len(sub) == 694
    assert set(sub.row_id) == set(sample.row_id) and sub.row_id.is_unique
    assert sub.prediction.notna().all() and sub.prediction.between(0, 1).all() and not sub.prediction.duplicated().any()
    sub.to_csv(f'{SUB}/{fname}', index=False)
    top = s.nlargest(139).index
    return {'파일': fname, '구성': desc,
            '21과 순위상관': round(pd.Series(s.values).corr(pd.Series(r21), method='spearman'), 3),
            '상위139 중 08-25 라벨=1': int((m.set_index('row_id').loc[top].prev == 1).sum())}

rows = [
    save(0.7 * r21 + 0.3 * r_rule, 'submit_36_ens21_prevrule30.csv', '0.7·21 + 0.3·규칙 (리더보드 0.6696)'),
    save(0.5 * r21 + 0.5 * r_rule, 'submit_38_ens21_prevrule50.csv', '0.5·21 + 0.5·규칙'),
    save(0.3 * r21 + 0.7 * r_rule, 'submit_39_ens21_prevrule70.csv', '0.3·21 + 0.7·규칙'),
    save(prev_f + 0.5 * r21,       'submit_40_prevgate_21order.csv', '라벨=1 그룹 우선, 안은 21 순서'),
]
display(pd.DataFrame(rows))

,파일,구성,21과 순위상관,상위139 중 08-25 라벨=1
0,submit_36_ens21_prevrule30.csv,0.7·21 + 0.3·규칙 (리더보드 0.6696),0.982,122
1,submit_38_ens21_prevrule50.csv,0.5·21 + 0.5·규칙,0.946,127
2,submit_39_ens21_prevrule70.csv,0.3·21 + 0.7·규칙,0.889,130
3,submit_40_prevgate_21order.csv,"라벨=1 그룹 우선, 안은 21 순서",0.984,134
